1. Import dan Konfigurasi

In [18]:
!pip install xlsxwriter -q

# %% 1. Import & konfigurasi
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xlsxwriter

# Mengatur direktori secara eksplisit untuk Google Colab agar tidak bentrok dengan sys.argv Jupyter
IN_DIR = Path('/content/drive/MyDrive/Colab Notebooks/Data_Mining/Bu_Karina/Dataset')
OUT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/Data_Mining/Bu_Karina/Dataset')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Parameter normalisasi dari dokumentasi dataset (UCI Bike Sharing)
TEMP_MIN, TEMP_MAX = -8, 39      # suhu (Celsius)        : (t - t_min) / (t_max - t_min)
ATEMP_MIN, ATEMP_MAX = -16, 50   # suhu terasa (Celsius) : (t - t_min) / (t_max - t_min)
HUM_SCALE = 100                  # kelembapan  : nilai asli / 100
WIND_SCALE = 67                  # kecepatan angin : nilai asli / 67

SEASON = {1: "Winter", 2: "Spring", 3: "Summer", 4: "Fall"}
WEATHER = {1: "Clear", 2: "Mist/Cloudy", 3: "Light Snow/Rain", 4: "Heavy Rain/Snow"}
WEEKDAY = {0: "Sunday", 1: "Monday", 2: "Tuesday", 3: "Wednesday",
           4: "Thursday", 5: "Friday", 6: "Saturday"}

DATA_DICT = {
    "date": ("dteday", "Tanggal pengamatan (YYYY-MM-DD)"),
    "hour": ("hr", "Jam pengamatan, 0-23"),
    "season": ("season", "Musim: Winter, Spring, Summer, Fall"),
    "year": ("yr", "Tahun: 2011 atau 2012"),
    "month": ("mnth", "Bulan, 1-12"),
    "weekday": ("weekday", "Nama hari (Sunday-Saturday)"),
    "is_holiday": ("holiday", "1 = hari libur, 0 = bukan"),
    "is_workingday": ("workingday", "1 = hari kerja (Senin-Jumat dan bukan libur), 0 = bukan"),
    "weather": ("weathersit", "Kondisi cuaca: Clear / Mist/Cloudy / Light Snow/Rain / Heavy Rain/Snow"),
    "temp_c": ("temp", "Suhu udara (derajat C) = temp x 47 - 8"),
    "feels_like_c": ("atemp", "Suhu terasa (derajat C) = atemp x 66 - 16"),
    "humidity_pct": ("hum", "Kelembapan (%) = hum x 100"),
    "windspeed": ("windspeed", "Kecepatan angin pada skala asli = windspeed x 67 "
                               "(satuan tidak dinyatakan pada dokumentasi sumber)"),
    "windspeed_imputed": ("(kolom baru)", "1 = windspeed asli bernilai 0 lalu diimputasi "
                                          "dengan median bulanan; 0 = nilai asli"),
    "casual": ("casual", "Jumlah peminjaman oleh pengguna tidak terdaftar"),
    "registered": ("registered", "Jumlah peminjaman oleh pengguna terdaftar"),
    "total_rentals": ("cnt", "Total peminjaman = casual + registered"),
}


2. Fungsi Bantu

In [19]:
# %% 2. Fungsi bantu
def iqr_outliers(s):
    """Jumlah nilai di luar batas 1.5 x IQR."""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())


def clean(raw):
    """Membersihkan satu dataset. Mengembalikan (data_bersih, log_cleansing)."""
    df = raw.copy()
    is_hour = "hr" in df.columns
    log = []

    def add(tahap, temuan, tindakan, baris=0):
        log.append({"No": len(log) + 1, "Tahap": tahap, "Temuan": temuan,
                    "Tindakan": tindakan, "Baris terdampak": int(baris)})

    add("Kondisi awal", f"{df.shape[0]:,} baris x {df.shape[1]} kolom", "-")

    # --- 1. Tipe data -------------------------------------------------------
    df["dteday"] = pd.to_datetime(df["dteday"])
    add("Tipe data", "dteday bertipe tanggal, kolom lain numerik",
        "Dipastikan ulang dengan pd.to_datetime; tidak ada perubahan berarti")

    # --- 2. Missing value ---------------------------------------------------
    n_null = int(df.isna().sum().sum())
    if n_null:
        df = df.dropna()
    add("Missing value", f"{n_null} sel kosong",
        "Baris yang mengandung nilai kosong dihapus" if n_null else "Tidak ada tindakan", n_null)

    # --- 3. Duplikat --------------------------------------------------------
    keys = ["dteday", "hr"] if is_hour else ["dteday"]
    n_dup = int(df.duplicated(keys).sum())
    if n_dup:
        df = df.drop_duplicates(keys, keep="first")
    add("Duplikat", f"{n_dup} baris duplikat berdasarkan kunci {keys}",
        "Duplikat dihapus (simpan yang pertama)" if n_dup else "Tidak ada tindakan", n_dup)

    # --- 4. Validasi konsistensi antar kolom --------------------------------
    d = df["dteday"]
    norm_cols = ["temp", "atemp", "hum", "windspeed"]
    checks = {
        "cnt = casual + registered": df["cnt"] != df["casual"] + df["registered"],
        "yr sesuai tahun pada tanggal": (d.dt.year - 2011) != df["yr"],
        "mnth sesuai bulan pada tanggal": d.dt.month != df["mnth"],
        "weekday sesuai hari pada tanggal": (d.dt.dayofweek + 1) % 7 != df["weekday"],
        "workingday = Senin-Jumat dan bukan libur":
            (df["weekday"].between(1, 5) & (df["holiday"] == 0)).astype(int) != df["workingday"],
        "season hanya bernilai 1-4": ~df["season"].isin(SEASON),
        "weathersit hanya bernilai 1-4": ~df["weathersit"].isin(WEATHER),
        "jumlah peminjaman tidak negatif": (df[["casual", "registered", "cnt"]] < 0).any(axis=1),
        "temp/atemp/hum/windspeed dalam rentang 0-1": ((df[norm_cols] < 0) | (df[norm_cols] > 1)).any(axis=1),
    }
    if is_hour:
        checks["hr dalam rentang 0-23"] = ~df["hr"].between(0, 23)
        checks["atribut kalender konstan dalam satu hari"] = (
            df.groupby("dteday")[["season", "yr", "mnth", "holiday", "weekday", "workingday"]]
            .transform("nunique") > 1).any(axis=1)
    for label, bad in checks.items():
        n = int(bad.sum())
        add("Validasi konsistensi", f"{label}: {n} baris melanggar",
            "Tidak perlu perbaikan" if n == 0 else "PERIKSA MANUAL", n)

    # --- 5. Anomali ---------------------------------------------------------
    # 5a. Kelembapan = 0
    m_hum = df["hum"] == 0
    n = int(m_hum.sum())
    tgl = ", ".join(sorted(df.loc[m_hum, "dteday"].dt.strftime("%Y-%m-%d").unique())) or "-"
    if n:
        hum = df["hum"].mask(m_hum)
        if is_hour:
            # interpolasi antar-hari pada jam yang sama (menjaga pola harian kelembapan)
            df["hum"] = hum.groupby(df["hr"]).transform(
                lambda s: s.interpolate(method="linear", limit_direction="both"))
        else:
            df["hum"] = hum.interpolate(method="linear", limit_direction="both")
    add("Anomali: humidity = 0",
        f"{n} baris (tanggal: {tgl}). Kelembapan 0% tidak realistis dan terkonsentrasi pada "
        "satu tanggal, indikasi sensor error" if n else "Tidak ditemukan",
        "Diubah menjadi NaN lalu diisi interpolasi linear antar-hari dari tanggal sebelum/sesudahnya"
        + (" (pada jam yang sama)" if is_hour else "") if n else "Tidak ada tindakan", n)

    # 5b. Kecepatan angin = 0
    m_ws = df["windspeed"] == 0
    n = int(m_ws.sum())
    if n:
        min_nonzero = df.loc[~m_ws, "windspeed"].min()
        nonzero = df["windspeed"].where(~m_ws)
        med = nonzero.groupby([df["yr"], df["mnth"]]).transform("median")
        df.loc[m_ws, "windspeed"] = med[m_ws]
        df["windspeed_imputed"] = m_ws.astype(int)
        add("Anomali: windspeed = 0",
            f"{n:,} baris ({n / len(df):.1%}) bernilai 0 dan tersebar di semua bulan. "
            f"Nilai non-nol terkecil adalah {min_nonzero:.4f}, sehingga 0 kemungkinan berarti "
            "'tidak terukur', bukan angin benar-benar tenang",
            "Diisi median windspeed non-nol pada tahun-bulan yang sama; ditandai pada kolom "
            "windspeed_imputed (1 = hasil imputasi)", n)
    else:
        add("Anomali: windspeed = 0", "Tidak ditemukan", "Tidak ada tindakan")

    # 5c. atemp = 0 (hanya dicek)
    n = int((df["atemp"] == 0).sum())
    add("Pengecekan: atemp = 0",
        f"{n} baris. Nilai 0 setara -16 C (batas bawah skala) dan konsisten dengan suhu "
        "sangat dingin pada baris di sekitarnya" if n else "Tidak ditemukan",
        "Dipertahankan (bukan error)" if n else "Tidak ada tindakan")

    # --- 6. Outlier (hanya dilaporkan) ---------------------------------------
    for col in ["cnt", "casual", "registered", "windspeed"]:
        n = iqr_outliers(df[col])
        add("Outlier (IQR 1.5x)", f"{col}: {n:,} baris ({n / len(df):.1%}) di luar batas IQR",
            "Dipertahankan: sesuai pola alami (jam sibuk, akhir pekan, cuaca ekstrem), "
            "bukan kesalahan input" if n else "Tidak ada tindakan")

    # --- 7. Kelengkapan jam (khusus hour) -------------------------------------
    if is_hour:
        n_days = df["dteday"].nunique()
        missing = n_days * 24 - len(df)
        short_days = int((df.groupby("dteday").size() < 24).sum())
        add("Kelengkapan jam",
            f"{missing} jam tidak tercatat (dari {n_days * 24:,} seharusnya), tersebar pada "
            f"{short_days} hari",
            "Dipertahankan apa adanya. Jam tanpa catatan kemungkinan berarti tanpa peminjaman; "
            "mengisi nol atau menebak cuaca berisiko menimbulkan bias", missing)

    # --- 8. Transformasi ----------------------------------------------------
    out = pd.DataFrame({"date": df["dteday"]})
    if is_hour:
        out["hour"] = df["hr"]
    out["season"] = pd.Categorical(df["season"].map(SEASON), list(SEASON.values()), ordered=True)
    out["year"] = df["yr"] + 2011
    out["month"] = df["mnth"]
    out["weekday"] = pd.Categorical(df["weekday"].map(WEEKDAY), list(WEEKDAY.values()), ordered=True)
    out["is_holiday"] = df["holiday"]
    out["is_workingday"] = df["workingday"]
    out["weather"] = pd.Categorical(df["weathersit"].map(WEATHER), list(WEATHER.values()), ordered=True)
    out["temp_c"] = (df["temp"] * (TEMP_MAX - TEMP_MIN) + TEMP_MIN).round(2)
    out["feels_like_c"] = (df["atemp"] * (ATEMP_MAX - ATEMP_MIN) + ATEMP_MIN).round(2)
    out["humidity_pct"] = (df["hum"] * HUM_SCALE).round(2)
    out["windspeed"] = (df["windspeed"] * WIND_SCALE).round(2)
    if "windspeed_imputed" in df:
        out["windspeed_imputed"] = df["windspeed_imputed"]
    out["casual"] = df["casual"]
    out["registered"] = df["registered"]
    out["total_rentals"] = df["cnt"]
    out = out.sort_values(["date", "hour"] if is_hour else ["date"]).reset_index(drop=True)

    add("Transformasi: kolom",
        "instant hanya nomor urut baris; nama kolom singkat sulit dibaca",
        "instant dihapus; kolom diganti nama (lihat sheet kamus_data)")
    add("Transformasi: dekode kategori",
        "season, weathersit, weekday, yr berupa kode angka",
        "Diubah menjadi label teks (tipe kategori berurutan); yr menjadi tahun 2011/2012")
    add("Transformasi: denormalisasi",
        "temp, atemp, hum, windspeed berskala 0-1",
        f"temp_c = temp x {TEMP_MAX - TEMP_MIN} {TEMP_MIN:+d}; "
        f"feels_like_c = atemp x {ATEMP_MAX - ATEMP_MIN} {ATEMP_MIN:+d}; "
        f"humidity_pct = hum x {HUM_SCALE}; windspeed = windspeed x {WIND_SCALE}; dibulatkan 2 desimal")

    # --- 9. Verifikasi akhir --------------------------------------------------
    assert out.isna().sum().sum() == 0, "Masih ada nilai kosong"
    assert (out["total_rentals"] == out["casual"] + out["registered"]).all()
    assert (out["humidity_pct"] > 0).all(), "Masih ada humidity = 0"
    add("Kondisi akhir", f"{out.shape[0]:,} baris x {out.shape[1]} kolom, tanpa nilai kosong", "-")

    return out, pd.DataFrame(log)


def write_xlsx(path, data, log_df, dict_df):
    """Tulis 3 sheet (data, log_cleansing, kamus_data) dengan format Arial."""
    wb = xlsxwriter.Workbook(path)
    base = {"font_name": "Arial", "font_size": 10}
    f_base = wb.add_format(base)
    f_date = wb.add_format({**base, "num_format": "yyyy-mm-dd"})
    f_wrap = wb.add_format({**base, "text_wrap": True, "valign": "top"})
    f_head = wb.add_format({**base, "bold": True, "bg_color": "#D9E1F2", "border": 1,
                            "text_wrap": True, "valign": "top"})

    def dump(df, sheet, widths=None, wrap=False):
        ws = wb.add_worksheet(sheet)
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, f_head)
            s = df[col]
            if col == "date":
                ws.write_column(1, i, [t.to_pydatetime() for t in s], f_date)
            else:
                ws.write_column(1, i, s.astype(object).tolist() if str(s.dtype) == "category"
                                else s.tolist(), f_wrap if wrap else f_base)
            if widths:
                w = widths[col]
            else:
                w = min(max(len(col), int(s.astype(str).str.len().max())) + 3, 40)
            ws.set_column(i, i, w)
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, len(df), len(df.columns) - 1)

    dump(data, "data")
    dump(log_df, "log_cleansing",
         {"No": 5, "Tahap": 28, "Temuan": 70, "Tindakan": 62, "Baris terdampak": 16}, wrap=True)
    dump(dict_df, "kamus_data", {"Kolom": 20, "Kolom asli": 16, "Keterangan": 85}, wrap=True)
    wb.close()


3. Ekspor kedua Dataset

In [20]:
# %% 3. Jalankan untuk kedua dataset
for name in ("day", "hour"):
    raw = pd.read_excel(IN_DIR / f"{name}.xlsx")
    clean_df, log_df = clean(raw)

    dict_df = pd.DataFrame(
        [{"Kolom": c, "Kolom asli": DATA_DICT[c][0], "Keterangan": DATA_DICT[c][1]}
         for c in clean_df.columns])

    out_path = OUT_DIR / f"{name}_clean.xlsx"
    write_xlsx(out_path, clean_df, log_df, dict_df)

    print(f"\n===== {name}.xlsx -> {out_path.name}: {raw.shape} -> {clean_df.shape}")
    print(log_df[["Tahap", "Temuan", "Baris terdampak"]].to_string(index=False, max_colwidth=90))


===== day.xlsx -> day_clean.xlsx: (731, 16) -> (731, 15)
                        Tahap                                                                                     Temuan  Baris terdampak
                 Kondisi awal                                                                       731 baris x 16 kolom                0
                    Tipe data                                                 dteday bertipe tanggal, kolom lain numerik                0
                Missing value                                                                               0 sel kosong                0
                     Duplikat                                              0 baris duplikat berdasarkan kunci ['dteday']                0
         Validasi konsistensi                                               cnt = casual + registered: 0 baris melanggar                0
         Validasi konsistensi                                            yr sesuai tahun pada tanggal: 0 baris mel